# Práctica 2 - Modelos Probabilísticos y Filtros Discretos
**I-402 - Principios de la Robótica Autónoma**

Universidad de San Andrés, Depto. de Ingeniería

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import timeit

---
## Ejercicio 1: Muestreo de distribuciones de probabilidad

Implementar tres funciones que generen muestras de una distribución normal $\mathcal{N}(\mu, \sigma^2)$ usando únicamente muestras de una distribución uniforme como fuente de aleatoriedad.

1. Sumando 12 distribuciones uniformes (Teorema Central del Límite)
2. Muestreo con rechazo
3. Transformación de Box-Muller: $x = \cos(2\pi u_1)\sqrt{-2\log u_2}$

### 1.1 Suma de 12 distribuciones uniformes

Se aprovecha el Teorema Central del Límite: la suma de 12 variables $U(0,1)$ tiene media 6 y varianza 1. Restando 6 se obtiene una variable con media 0 y varianza 1, aproximadamente normal.

In [ ]:
def normal_suma_uniformes(mu, sigma2, n=1):
    """Genera muestras normales sumando 12 distribuciones uniformes."""
    sigma = np.sqrt(sigma2)
    samples = np.zeros(n)
    for i in range(n):
        s = np.sum(np.random.uniform(0, 1, 12)) - 6  # media 0, var 1
        samples[i] = mu + sigma * s
    return samples

### 1.2 Muestreo con rechazo

Se propone una muestra $x$ de una distribución uniforme en $[\mu - 5\sigma, \mu + 5\sigma]$ y se acepta con probabilidad proporcional a la densidad gaussiana evaluada en $x$. Se rechaza y se vuelve a muestrear si no se acepta.

In [ ]:
def normal_rechazo(mu, sigma2, n=1):
    """Genera muestras normales usando muestreo con rechazo."""
    sigma = np.sqrt(sigma2)
    samples = []
    while len(samples) < n:
        x = np.random.uniform(mu - 5*sigma, mu + 5*sigma)
        u = np.random.uniform(0, 1)
        fx = np.exp(-0.5 * ((x - mu) / sigma) ** 2)
        if u <= fx:
            samples.append(x)
    return np.array(samples)

### 1.3 Transformación de Box-Muller

Dadas dos muestras uniformes $u_1, u_2 \in [0, 1]$, se genera una muestra normal estándar mediante:

$$x = \cos(2\pi u_1) \sqrt{-2 \log u_2}$$

In [ ]:
def normal_box_muller(mu, sigma2, n=1):
    """Genera muestras normales usando Box-Muller."""
    sigma = np.sqrt(sigma2)
    samples = np.zeros(n)
    for i in range(n):
        u1 = np.random.uniform(0, 1)
        u2 = np.random.uniform(0, 1)
        x = np.cos(2 * np.pi * u1) * np.sqrt(-2 * np.log(u2))
        samples[i] = mu + sigma * x
    return samples

### Comparación de histogramas

Se generan 10000 muestras con cada método y se comparan con la densidad teórica $\mathcal{N}(0, 1)$.

In [ ]:
mu, sigma2 = 0, 1
N = 10000

methods = [
    ("Suma 12 uniformes", normal_suma_uniformes(mu, sigma2, N)),
    ("Muestreo con rechazo", normal_rechazo(mu, sigma2, N)),
    ("Box-Muller", normal_box_muller(mu, sigma2, N)),
    ("numpy.random.normal", np.random.normal(mu, np.sqrt(sigma2), N)),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
x_teo = np.linspace(-4, 4, 200)
y_teo = (1/np.sqrt(2*np.pi*sigma2)) * np.exp(-0.5*(x_teo - mu)**2 / sigma2)

for ax, (name, samples) in zip(axes.flat, methods):
    ax.hist(samples, bins=50, density=True, alpha=0.7, label=name)
    ax.plot(x_teo, y_teo, 'r-', lw=2, label='N(0,1) teórica')
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()

### Comparación de tiempos de ejecución

Se usa `timeit` para medir el tiempo de generar 10000 muestras con cada método (10 repeticiones).

In [ ]:
methods_fn = [
    ("Suma 12 uniformes", lambda: normal_suma_uniformes(mu, sigma2, N)),
    ("Muestreo con rechazo", lambda: normal_rechazo(mu, sigma2, N)),
    ("Box-Muller", lambda: normal_box_muller(mu, sigma2, N)),
    ("numpy.random.normal", lambda: np.random.normal(mu, np.sqrt(sigma2), N)),
]

print("Comparación de tiempos (10000 muestras, 10 repeticiones)")
print("-" * 55)
for name, fn in methods_fn:
    t = timeit.timeit(fn, number=10)
    print(f"{name:25s}: {t:.4f} s")

**Análisis de tiempos:**

- **Box-Muller** es el más rápido de los tres métodos implementados, ya que solo requiere 2 muestras uniformes por muestra normal.
- **Suma de 12 uniformes** es algo más lento porque necesita generar 12 muestras uniformes por cada muestra normal.
- **Muestreo con rechazo** es el más lento porque muchas muestras propuestas son rechazadas (la tasa de aceptación depende de la relación entre la envolvente uniforme y la gaussiana).
- **numpy.random.normal** es órdenes de magnitud más rápido porque está implementado en C optimizado y opera de forma vectorizada.

---
## Ejercicio 2: Modelo de movimiento basado en odometría

Implementar un modelo de movimiento basado en odometría con ruido. Los argumentos son:

- $\mathbf{x}_t = [x, y, \theta]^T$: pose actual
- $\mathbf{u}_t = [\delta_{r1}, \delta_{r2}, \delta_t]^T$: lectura de odometría
- $\alpha = [\alpha_1, \alpha_2, \alpha_3, \alpha_4]^T$: parámetros de ruido

El modelo agrega ruido normal a cada componente de la odometría:

$$\hat{\delta}_{r1} = \delta_{r1} + \mathcal{N}(0, \alpha_1 \delta_{r1}^2 + \alpha_2 \delta_t^2)$$
$$\hat{\delta}_t = \delta_t + \mathcal{N}(0, \alpha_3 \delta_{r1}^2 + \alpha_3 \delta_{r2}^2 + \alpha_4 \delta_t^2)$$
$$\hat{\delta}_{r2} = \delta_{r2} + \mathcal{N}(0, \alpha_1 \delta_{r2}^2 + \alpha_2 \delta_t^2)$$

Y luego se calcula la nueva pose:

$$x' = x + \hat{\delta}_t \cos(\theta + \hat{\delta}_{r1})$$
$$y' = y + \hat{\delta}_t \sin(\theta + \hat{\delta}_{r1})$$
$$\theta' = \theta + \hat{\delta}_{r1} + \hat{\delta}_{r2}$$

In [ ]:
def odometry_motion_model(xt, ut, alpha):
    """
    Modelo de movimiento basado en odometría.
    xt: [x, y, theta]
    ut: [delta_r1, delta_r2, delta_t]
    alpha: [alpha1, alpha2, alpha3, alpha4]
    """
    x, y, theta = xt
    delta_r1, delta_r2, delta_t = ut
    a1, a2, a3, a4 = alpha

    # Agregar ruido normal a los comandos de odometría
    delta_r1_hat = delta_r1 + normal_box_muller(0, a1 * delta_r1**2 + a2 * delta_t**2)[0]
    delta_t_hat  = delta_t  + normal_box_muller(0, a3 * delta_r1**2 + a3 * delta_r2**2 + a4 * delta_t**2)[0]
    delta_r2_hat = delta_r2 + normal_box_muller(0, a1 * delta_r2**2 + a2 * delta_t**2)[0]

    # Calcular nueva pose
    x_new = x + delta_t_hat * np.cos(theta + delta_r1_hat)
    y_new = y + delta_t_hat * np.sin(theta + delta_r1_hat)
    theta_new = theta + delta_r1_hat + delta_r2_hat

    return np.array([x_new, y_new, theta_new])

### Evaluación con 5000 muestras

Parámetros:
$$\mathbf{x}_t = \begin{pmatrix} 2.0 \\ 4.0 \\ 0.0 \end{pmatrix} \quad \mathbf{u}_t = \begin{pmatrix} \pi/2 \\ 0.0 \\ 1.0 \end{pmatrix} \quad \alpha = \begin{pmatrix} 0.1 \\ 0.1 \\ 0.01 \\ 0.01 \end{pmatrix}$$

In [ ]:
xt = np.array([2.0, 4.0, 0.0])
ut = np.array([np.pi/2, 0.0, 1.0])  # [delta_r1, delta_r2, delta_t]
alpha = np.array([0.1, 0.1, 0.01, 0.01])

N_samples = 5000
samples = np.array([odometry_motion_model(xt, ut, alpha) for _ in range(N_samples)])

plt.figure(figsize=(8, 8))
plt.scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.5, label='Muestras')
plt.plot(xt[0], xt[1], 'ro', markersize=8, label='Pose inicial')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Modelo de movimiento basado en odometría (5000 muestras)')
plt.legend()
plt.axis('equal')
plt.grid(True)
plt.tight_layout()
plt.show()

**Observación:** La nube de puntos muestra la incertidumbre en la pose futura del robot. Con $\delta_{r1} = \pi/2$ el robot primero rota 90° y luego avanza 1 unidad (hacia arriba desde la pose inicial). El ruido en las rotaciones genera una dispersión en forma de arco, mientras que el ruido en la traslación genera dispersión radial.

---
## Ejercicio 3: Modelo de movimiento en ROS2

La función `odometry_motion_model` en `tp2/robot_functions.py` fue completada con el mismo modelo de ruido. El código implementado es:

```python
def odometry_motion_model(self, xt, ut, alpha):
    x, y, theta = xt
    delta_r1, delta_t, delta_r2 = ut  # Nota: orden distinto (r1, t, r2) en ROS
    a1, a2, a3, a4 = alpha

    delta_r1_hat = delta_r1 + np.random.normal(0, np.sqrt(abs(a1*delta_r1**2 + a2*delta_t**2) + 1e-10))
    delta_t_hat  = delta_t  + np.random.normal(0, np.sqrt(abs(a3*delta_r1**2 + a3*delta_r2**2 + a4*delta_t**2) + 1e-10))
    delta_r2_hat = delta_r2 + np.random.normal(0, np.sqrt(abs(a1*delta_r2**2 + a2*delta_t**2) + 1e-10))

    x_new = x + delta_t_hat * np.cos(theta + delta_r1_hat)
    y_new = y + delta_t_hat * np.sin(theta + delta_r1_hat)
    theta_new = theta + delta_r1_hat + delta_r2_hat

    return np.array([x_new, y_new, theta_new])
```

Para ejecutarlo en ROS2:
```bash
# Terminal 1:
ros2 launch turtlebot3_custom_simulation custom_simulation.launch.py
# Terminal 2:
ros2 launch tp2 launch_my_odometry.launch.py
# Terminal 3:
ros2 run turtlebot3_teleop teleop_keyboard
```

Para variar los alpha:
```bash
ros2 launch tp2 launch_my_odometry.launch.py alpha:="[0.5, 0.5, 0.05, 0.05]"
```

**Efecto de variar los alpha:**
- $\alpha_1, \alpha_2$ altos → mayor incertidumbre en las rotaciones → la nube se dispersa en forma de arco.
- $\alpha_3, \alpha_4$ altos → mayor incertidumbre en la traslación → la nube se alarga radialmente.

---
## Ejercicio 4: Filtro Discreto de Bayes

Mundo unidimensional de 20 celdas. Robot inicialmente en celda 10.

**Modelo de movimiento (avanzar):**
- 25% quedarse en la misma celda
- 50% avanzar 1 celda
- 25% avanzar 2 celdas

**Bordes:**
- Última celda: 100% quedarse
- Penúltima celda: 25% quedarse, 75% avanzar 1

Retroceder es análogo en sentido opuesto.

Se ejecutan 9 comandos de avanzar + 3 de retroceder.

In [ ]:
N_CELDAS = 20

def predict(bel, command):
    """Predicción del filtro de Bayes discreto."""
    new_bel = np.zeros(N_CELDAS)

    for j in range(N_CELDAS):
        if command == 'avanzar':
            if j == N_CELDAS - 1:
                # Última celda: 100% quedarse
                new_bel[j] += bel[j]
            elif j == N_CELDAS - 2:
                # Penúltima: 25% quedarse, 75% avanzar 1
                new_bel[j]     += 0.25 * bel[j]
                new_bel[j + 1] += 0.75 * bel[j]
            else:
                # Normal: 25% quedarse, 50% +1, 25% +2
                new_bel[j]     += 0.25 * bel[j]
                new_bel[j + 1] += 0.50 * bel[j]
                new_bel[j + 2] += 0.25 * bel[j]

        elif command == 'retroceder':
            if j == 0:
                # Primera celda: 100% quedarse
                new_bel[j] += bel[j]
            elif j == 1:
                # Segunda celda: 25% quedarse, 75% retroceder 1
                new_bel[j]     += 0.25 * bel[j]
                new_bel[j - 1] += 0.75 * bel[j]
            else:
                # Normal: 25% quedarse, 50% -1, 25% -2
                new_bel[j]     += 0.25 * bel[j]
                new_bel[j - 1] += 0.50 * bel[j]
                new_bel[j - 2] += 0.25 * bel[j]

    return new_bel

In [ ]:
# Belief inicial: robot en celda 10
bel = np.hstack((np.zeros(10), 1, np.zeros(9)))

# 9 comandos de avanzar
for _ in range(9):
    bel = predict(bel, 'avanzar')

# 3 comandos de retroceder
for _ in range(3):
    bel = predict(bel, 'retroceder')

# Graficar
plt.figure(figsize=(10, 5))
plt.bar(range(N_CELDAS), bel, color='steelblue', edgecolor='black')
plt.xlabel('Celda')
plt.ylabel('Probabilidad (belief)')
plt.title('Belief del robot después de 9 avances y 3 retrocesos')
plt.xticks(range(N_CELDAS))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Mostrar valores
print("Belief final:")
for i, b in enumerate(bel):
    if b > 0.001:
        print(f"  Celda {i}: {b:.4f}")
print(f"Suma total: {np.sum(bel):.6f}")

**Análisis del resultado:**

El belief se distribuye aproximadamente entre las celdas 9 y 19, con el pico en la celda 16. Esto tiene sentido:
- El robot parte de la celda 10 y ejecuta 9 avances → el valor esperado del movimiento neto es $9 \times 0.75 = 6.75$ celdas hacia adelante (ya que el avance promedio es $0.25 \times 0 + 0.5 \times 1 + 0.25 \times 2 = 1$ pero se ve limitado por los bordes).
- Luego retrocede 3 pasos → movimiento esperado de vuelta: $3 \times 1 = 3$ celdas.
- El efecto de los bordes (celda 19 absorbe probabilidad) hace que la distribución se acumule cerca del borde derecho y no sea simétrica.
- La incertidumbre crece con cada paso debido al modelo estocástico de movimiento.